In [1]:
# store × product × context
#         ↓
# engineered relevance features
#         ↓
# promotion opportunity score
#         ↓
# top products per Walmart

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
INPUT_PATH = Path(
    "data/processed/gta_promotion_context_v1.parquet"
)

OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = (
    OUTPUT_DIR /
    "gta_promotion_rankings_v1.parquet"
)

In [4]:
df = pd.read_parquet(INPUT_PATH)

print(df.shape)
print(df.columns.tolist())

display(df.head())

(105, 44)
['store_id', 'store_name', 'address', 'city', 'province', 'postal_code', 'full_address', 'latitude', 'longitude', 'universities', 'colleges', 'schools', 'offices', 'industrial', 'commercial', 'retail_landuse', 'rail_stations', 'store_name_demo', 'population_5km', 'population_density_5km', 'median_household_income_5km', 'average_household_size_5km', 'employment_rate_5km', 'n_census_tracts', 'product_id', 'product_name', 'category', 'current_price', 'regular_price', 'savings', 'promotion_type', 'product_url', 'page_number', 'snapshot_date', 'country', 'region', 'source', 'captured_at', 'discount_amount', 'discount_pct', 'is_promoted', 'promo_id', 'promo_price', 'context_id']


,store_id,store_name,address,city,province,postal_code,full_address,latitude,longitude,universities,...,country,region,source,captured_at,discount_amount,discount_pct,is_promoted,promo_id,promo_price,context_id
0,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.65598,-79.435728,19,...,CA,None,Walmart Canada Weekly Flyer,2026-09-11 21:24:53.568645+00:00,0.50,NaN,1,49af3e3b67cadc003a9f,97.00,5c8f6edd755e34fde827
1,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.65598,-79.435728,19,...,CA,None,Walmart Canada Weekly Flyer,2026-09-11 21:24:53.568645+00:00,1.00,25.188917,1,96bf60d742227c0d20e5,2.97,80eb63cb821d22687b8c
2,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.65598,-79.435728,19,...,CA,None,Walmart Canada Weekly Flyer,2026-09-11 21:24:53.568645+00:00,2.47,18.377976,1,d8c9483775d8bacb07df,10.97,bd79ba49ce5c202840cb
3,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.65598,-79.435728,19,...,CA,None,Walmart Canada Weekly Flyer,2026-09-11 21:24:53.568645+00:00,5.61,22.025913,1,30fefe29f8be87fd5177,19.86,cae9459736ebe7205a48
4,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.65598,-79.435728,19,...,CA,None,Walmart Canada Weekly Flyer,2026-09-11 21:24:53.568645+00:00,3.99,40.020060,1,f9d6443ae13571f5c247,5.98,33aa90af8aba68777349


In [5]:
print("Stores:", df["store_id"].nunique())

print(
    "Products:",
    df["product_id"].nunique()
    if "product_id" in df.columns
    else df["product_name"].nunique()
)

print(
    "Rows:",
    len(df)
)

Stores: 15
Products: 7
Rows: 105


# Build score components
discount strength

local population

local income

commercial/retail activity

weather context

# Helper scaling function

In [6]:
def minmax(series):

    s = pd.to_numeric(
        series,
        errors="coerce"
    )

    if s.notna().sum() == 0:
        return pd.Series(
            0.0,
            index=series.index
        )

    min_v = s.min()
    max_v = s.max()

    if max_v == min_v:
        return pd.Series(
            0.5,
            index=series.index
        )

    return (
        (s - min_v)
        / (max_v - min_v)
    ).fillna(0)

# Discount score

In [7]:
df["discount_score"] = minmax(
    df["discount_pct"]
    .clip(lower=0, upper=60)
)

# Population score

In [9]:
if "population_5km" in df.columns:

    df["population_score"] = minmax(
        df["population_5km"]
    )

else:
    df["population_score"] = 0.5

# Income score

In [10]:
if "median_household_income_5km" in df.columns:

    df["income_score"] = minmax(
        df["median_household_income_5km"]
    )

else:
    df["income_score"] = 0.5

# Commercial activity score

In [11]:
commercial_features = [
    c for c in [
        "commercial",
        "retail_landuse",
        "offices"
    ]
    if c in df.columns
]

if commercial_features:

    df["commercial_raw"] = (
        df[commercial_features]
        .fillna(0)
        .sum(axis=1)
    )

    df["commercial_score"] = minmax(
        df["commercial_raw"]
    )

else:
    df["commercial_score"] = 0.5

# Weather relevance

In [12]:
df["weather_score"] = 0.0

In [13]:
if "mean_temperature" in df.columns:

    df["temp_extreme"] = (
        np.abs(
            df["mean_temperature"] - 15
        )
        / 25
    ).clip(0, 1)

    df["weather_score"] += (
        0.5 * df["temp_extreme"]
    )

In [14]:
if "total_rain" in df.columns:

    rain_score = minmax(
        df["total_rain"]
        .fillna(0)
        .clip(upper=30)
    )

    df["weather_score"] += (
        0.25 * rain_score
    )

In [15]:
if "total_snow" in df.columns:

    snow_score = minmax(
        df["total_snow"]
        .fillna(0)
        .clip(upper=20)
    )

    df["weather_score"] += (
        0.25 * snow_score
    )

In [16]:
df["weather_score"] = (
    df["weather_score"]
    .clip(0, 1)
)

# Promotion opportunity score

In [17]:
WEIGHTS = {
    "discount_score": 0.45,
    "population_score": 0.20,
    "commercial_score": 0.15,
    "income_score": 0.10,
    "weather_score": 0.10
}

In [18]:
df["opportunity_score"] = (
    WEIGHTS["discount_score"]
    * df["discount_score"]

    + WEIGHTS["population_score"]
    * df["population_score"]

    + WEIGHTS["commercial_score"]
    * df["commercial_score"]

    + WEIGHTS["income_score"]
    * df["income_score"]

    + WEIGHTS["weather_score"]
    * df["weather_score"]
)

In [19]:
df["opportunity_score"] = (
    df["opportunity_score"]
    * 100
)

# Rank within each store

In [20]:
df["store_rank"] = (
    df.groupby("store_id")[
        "opportunity_score"
    ]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

# Top 10 per store

In [21]:
top10 = (
    df[
        df["store_rank"] <= 10
    ]
    .sort_values(
        [
            "store_id",
            "store_rank"
        ]
    )
)

In [22]:
cols = [
    c for c in [
        "store_id",
        "store_name",
        "product_name",
        "category",
        "promo_price",
        "regular_price",
        "discount_pct",
        "opportunity_score",
        "store_rank"
    ]
    if c in top10.columns
]

display(
    top10[cols].head(50)
)

,store_id,store_name,product_name,category,promo_price,regular_price,discount_pct,opportunity_score,store_rank
18,1004,Toronto Stockyards,"Janes® Pub Style Chicken Strips, 700g, Frozen ...",Grocery,5.98,9.97,40.020060,72.395502,1
15,1004,Toronto Stockyards,"EGGO Blueberry Flavour Waffles, 280g (8 waffle...",Grocery,2.97,3.97,25.188917,41.557370,2
17,1004,Toronto Stockyards,Tim Hortons Original Blend Medium Roast Coffee...,Grocery,19.86,25.47,22.025913,34.980592,3
14,1004,Toronto Stockyards,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,97.00,NaN,NaN,27.395502,4
16,1004,Toronto Stockyards,Quaker Chewy Chocolate Chips Granola Bars Vari...,Grocery,10.97,13.44,18.377976,27.395502,5
19,1004,Toronto Stockyards,Sealtest Chocolate Partly Skimmed 1% Milk 750ml,Grocery,1.93,NaN,NaN,27.395502,6
20,1004,Toronto Stockyards,"Red Bull Energy Drink, 250 ml (4 pack)",Grocery,10.97,NaN,NaN,27.395502,7
53,1061,Mississauga Heartland,"Janes® Pub Style Chicken Strips, 700g, Frozen ...",Grocery,5.98,9.97,40.020060,60.866870,1
50,1061,Mississauga Heartland,"EGGO Blueberry Flavour Waffles, 280g (8 waffle...",Grocery,2.97,3.97,25.188917,30.028738,2
52,1061,Mississauga Heartland,Tim Hortons Original Blend Medium Roast Coffee...,Grocery,19.86,25.47,22.025913,23.451960,3


# Explainability

In [23]:
df["discount_contribution"] = (
    WEIGHTS["discount_score"]
    * df["discount_score"]
    * 100
)

df["population_contribution"] = (
    WEIGHTS["population_score"]
    * df["population_score"]
    * 100
)

df["commercial_contribution"] = (
    WEIGHTS["commercial_score"]
    * df["commercial_score"]
    * 100
)

df["income_contribution"] = (
    WEIGHTS["income_score"]
    * df["income_score"]
    * 100
)

df["weather_contribution"] = (
    WEIGHTS["weather_score"]
    * df["weather_score"]
    * 100
)

# QA

In [24]:
print(
    df["opportunity_score"]
    .describe()
)

count    105.000000
mean      26.482996
std       17.359797
min        7.646807
25%       13.459188
50%       20.530295
75%       31.490611
max       81.131981
Name: opportunity_score, dtype: float64


In [25]:
print(
    "Min:",
    df["opportunity_score"].min()
)

print(
    "Max:",
    df["opportunity_score"].max()
)

Min: 7.646806591885327
Max: 81.13198059640155


# Check ranking

In [26]:
for store_id in (
    df["store_id"]
    .drop_duplicates()
    .head(3)
):

    print("\nSTORE:", store_id)

    display(
        df[
            df["store_id"] == store_id
        ]
        .sort_values(
            "store_rank"
        )[cols]
        .head(10)
    )


STORE: 3106


,store_id,store_name,product_name,category,promo_price,regular_price,discount_pct,opportunity_score,store_rank
4,3106,Toronto Dufferin Mall,"Janes® Pub Style Chicken Strips, 700g, Frozen ...",Grocery,5.98,9.97,40.020060,81.131981,1
1,3106,Toronto Dufferin Mall,"EGGO Blueberry Flavour Waffles, 280g (8 waffle...",Grocery,2.97,3.97,25.188917,50.293848,2
3,3106,Toronto Dufferin Mall,Tim Hortons Original Blend Medium Roast Coffee...,Grocery,19.86,25.47,22.025913,43.717070,3
0,3106,Toronto Dufferin Mall,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,97.00,NaN,NaN,36.131981,4
2,3106,Toronto Dufferin Mall,Quaker Chewy Chocolate Chips Granola Bars Vari...,Grocery,10.97,13.44,18.377976,36.131981,5
5,3106,Toronto Dufferin Mall,Sealtest Chocolate Partly Skimmed 1% Milk 750ml,Grocery,1.93,NaN,NaN,36.131981,6
6,3106,Toronto Dufferin Mall,"Red Bull Energy Drink, 250 ml (4 pack)",Grocery,10.97,NaN,NaN,36.131981,7



STORE: 1139


,store_id,store_name,product_name,category,promo_price,regular_price,discount_pct,opportunity_score,store_rank
11,1139,Toronto Downsview,"Janes® Pub Style Chicken Strips, 700g, Frozen ...",Grocery,5.98,9.97,40.020060,55.631655,1
8,1139,Toronto Downsview,"EGGO Blueberry Flavour Waffles, 280g (8 waffle...",Grocery,2.97,3.97,25.188917,24.793523,2
10,1139,Toronto Downsview,Tim Hortons Original Blend Medium Roast Coffee...,Grocery,19.86,25.47,22.025913,18.216745,3
7,1139,Toronto Downsview,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,97.00,NaN,NaN,10.631655,4
9,1139,Toronto Downsview,Quaker Chewy Chocolate Chips Granola Bars Vari...,Grocery,10.97,13.44,18.377976,10.631655,5
12,1139,Toronto Downsview,Sealtest Chocolate Partly Skimmed 1% Milk 750ml,Grocery,1.93,NaN,NaN,10.631655,6
13,1139,Toronto Downsview,"Red Bull Energy Drink, 250 ml (4 pack)",Grocery,10.97,NaN,NaN,10.631655,7



STORE: 1004


,store_id,store_name,product_name,category,promo_price,regular_price,discount_pct,opportunity_score,store_rank
18,1004,Toronto Stockyards,"Janes® Pub Style Chicken Strips, 700g, Frozen ...",Grocery,5.98,9.97,40.020060,72.395502,1
15,1004,Toronto Stockyards,"EGGO Blueberry Flavour Waffles, 280g (8 waffle...",Grocery,2.97,3.97,25.188917,41.557370,2
17,1004,Toronto Stockyards,Tim Hortons Original Blend Medium Roast Coffee...,Grocery,19.86,25.47,22.025913,34.980592,3
14,1004,Toronto Stockyards,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,97.00,NaN,NaN,27.395502,4
16,1004,Toronto Stockyards,Quaker Chewy Chocolate Chips Granola Bars Vari...,Grocery,10.97,13.44,18.377976,27.395502,5
19,1004,Toronto Stockyards,Sealtest Chocolate Partly Skimmed 1% Milk 750ml,Grocery,1.93,NaN,NaN,27.395502,6
20,1004,Toronto Stockyards,"Red Bull Energy Drink, 250 ml (4 pack)",Grocery,10.97,NaN,NaN,27.395502,7


# Save full rankings

In [27]:
df.to_parquet(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)

Saved: results/gta_promotion_rankings_v1.parquet


In [28]:
TOP_PATH = (
    OUTPUT_DIR /
    "gta_top10_promotions_per_store_v1.parquet"
)

top10.to_parquet(
    TOP_PATH,
    index=False
)

print("Saved:", TOP_PATH)

Saved: results/gta_top10_promotions_per_store_v1.parquet
